# Prompt Versioning과 Rollback

당신이 PM이고 제품 지원 시스템이 LLM을 사용하여 들어오는 ticket을 적절한 팀으로 전달한다고 상상해 보세요. API 관련 ticket을 더 많이 platform 팀으로 보내도록 routing prompt를 조정하려고 합니다. 이전에는 prompt가 codebase에 있었습니다. 변경하려면 PR, CI 실행, deployment가 필요했습니다. 되돌릴 때도 같은 변경이 필요했습니다. 대신 Managed Agents는 prompt를 서버 측에 보관합니다. 모든 `agents.update`는 새로운 immutable version을 만들며 세션은 ID로 사용할 version을 선택합니다. 여전히 prompt 변경을 검토하고 승인하지만, 코드 diff 대신 config의 version 번호를 승인하며, 실행 중인 서비스는 rebuild 없이 변경을 반영합니다. 문제가 생기면 caller가 이전 version을 가리키게 하는 것만으로 rollback할 수 있습니다.

이 cookbook에서는 지원 ticket 분류 agent를 만들고, system prompt를 update한 뒤, 성능이 저하되면 rollback하겠습니다.

마칠 때면 다음을 수행한 상태입니다.
- agent를 생성하고 `version: 1`로 반환되는 것을 확인하기
- labelled test set으로 특정 version 평가하기
- v2 prompt를 출시하고 version 번호가 바뀌는 것 확인하기
- deploy 없이 성능이 저하된 agent를 v1으로 rollback하기

## 사전 요구 사항

**필수:**
- Python 3.11+
- `anthropic>=0.91.0`
- Anthropic API key
- 이 디렉터리의 `example_data/prompt_versioning_and_rollback/support_tickets.jsonl` fixture


## 설정


In [1]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import json
import os
import time
from collections import defaultdict
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in your environment or .env file")

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = Anthropic()

## agent 생성하기(version 1)

먼저 Environment(agent가 실행되는 container template)와 agent 자체를 생성하겠습니다. system prompt는 짧습니다. 각 ticket을 팀과 우선순위로 분류하고 JSON으로만 응답합니다.


In [3]:
env = client.beta.environments.create(name="ticket-triage-env")
ENV_ID = env.id

V1_SYSTEM = """You are a support-ticket triage agent for a usage-billed API product.
Read the ticket and respond with ONLY a single line of raw JSON (no code fences, no prose):
{"team": "<billing|auth|api-platform|dashboard>", "priority": "<P1|P2|P3>"}
Route based on the customer's actual problem, not surface keywords."""

agent = client.beta.agents.create(
    name="ticket-triage",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-prompt-versioning-and-rollback"},
    model=MODEL,
    system=V1_SYSTEM,
)
AGENT_ID = agent.id
print(f"env={ENV_ID}  agent={AGENT_ID} v{agent.version}")

env=env_017FKqTdU6dFgEe6zbbVQvE5  agent=agent_011CZo6t9Cr6Eg5TeFFPWPpi v1


요청하지 않았는데도 `agents.create`는 `version: 1`을 반환했습니다. 이후 수행하는 모든 update는 같은 `id`를 가진 새 version을 만들며, 세션 고정과 rollback에 이 번호를 사용합니다.


## labelled test set 불러오기

아래에는 팀별 다섯 개씩 ticket 스무 개가 있으며, 올바른 routing이 이미 label로 지정되어 있습니다. 평가의 기준이 되는 ground truth입니다.


In [4]:
fixture = Path("example_data/prompt_versioning_and_rollback/support_tickets.jsonl")
tickets = [json.loads(line) for line in fixture.read_text().splitlines()]
teams = sorted({t["team"] for t in tickets})
print(f"{len(tickets)} tickets across {len(teams)} teams: {teams}")

20 tickets across 4 teams: ['api-platform', 'auth', 'billing', 'dashboard']


## version 1 평가하기

모든 ticket을 v1으로 실행하고 결과를 보겠습니다. 아래 `triage` helper는 특정 version으로 고정한 세션을 열고, ticket 하나를 보내고, 세션이 idle이 될 때까지 `events.list`를 polling한 다음, `agent.message` event에서 JSON 판정을 parsing합니다.

이 cookbook에서 중요한 부분은 `sessions.create`의 `agent=` 인자입니다. 일반 문자열(`agent=AGENT_ID`)을 전달하면 그때의 최신 version을 사용합니다. `{"type": "agent", "id": ..., "version": ...}`를 전달하면 정확한 version으로 고정하며, 이는 통제된 비교에 원하는 방식입니다.


In [5]:
def triage(version: int, ticket: dict) -> dict:
    """고정된 agent version으로 ticket 하나를 실행하고 판정을 반환합니다."""
    session = client.beta.sessions.create(
        agent={"type": "agent", "id": AGENT_ID, "version": version},
        environment_id=ENV_ID,
    )
    try:
        prompt = "Subject: " + ticket["subject"] + "\n\n" + ticket["body"]
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": prompt}]}],
        )
        deadline = time.time() + 60
        while time.time() < deadline:
            events = client.beta.sessions.events.list(session.id).data
            if events and events[-1].type == "session.status_idle":
                break
            time.sleep(1)
        else:
            raise TimeoutError(f"session {session.id} did not idle within 60s")
        agent_events = [e for e in events if e.type == "agent.message"]
        reply = "".join(b.text for e in agent_events for b in e.content)
        return json.loads(reply)
    finally:
        try:
            client.beta.sessions.archive(session.id)
        except Exception:  # noqa: S110
            pass


def score(version: int) -> dict:
    """지정된 version으로 모든 ticket을 평가하고 팀별 정확도를 반환합니다."""
    hits = defaultdict(lambda: [0, 0])
    for t in tickets:
        pred = triage(version, t)
        hits[t["team"]][1] += 1
        if pred.get("team") == t["team"]:
            hits[t["team"]][0] += 1
    return dict(hits)


v1_scores = score(version=1)
print("v1 results:")
for team, (correct, total) in sorted(v1_scores.items()):
    print(f"  {team:14s} {correct}/{total}")

v1 results:
  api-platform   5/5
  auth           5/5
  billing        4/5
  dashboard      5/5


model은 잘 수행했고 거의 모든 routing을 정확히 처리했습니다.


## version 2 출시하기

이제 PM이 변경을 출시합니다. API 사용량이나 rate limit에 관한 모든 것이 platform 팀에 속한다는 routing 규칙입니다. `agents.update`는 agent를 새 system prompt로 update합니다.


In [6]:
V2_SYSTEM = V1_SYSTEM + (
    "\n\nROUTING RULE: If the ticket text mentions API usage, rate limits, quotas, "
    "or request volume, route to api-platform. Apply this rule before any other "
    "consideration; do not second-guess it based on the rest of the ticket."
)
agent = client.beta.agents.update(AGENT_ID, version=agent.version, system=V2_SYSTEM)
print(f"agent {AGENT_ID} now at v{agent.version}")

agent agent_011CZo6t9Cr6Eg5TeFFPWPpi now at v2


### code review는 어디로 갔나요?

API 호출 한 번으로 agent를 변경했습니다. 이 notebook에서는 아무 검토 없이 이루어졌으며 demo로는 괜찮지만 production 운영 방식은 아닙니다.

`agents.update`에는 built-in 승인 workflow가 없습니다. workspace의 어떤 key든 호출할 수 있습니다. caller가 고정된 version 대신 agent ID만 전달한다면 바로 다음 세션부터 새 prompt를 사용합니다. deploy가 필요 없는 대신 치르는 대가이며, 코드 대신 API로 관리하는 feature flag나 다른 config에서 하는 것과 같은 선택입니다.

검토 단계를 다시 넣는 패턴은 다음과 같습니다. production caller는 항상 명시적인 version으로 고정하며, 변경 관리의 대상은 *그 고정된 번호*입니다. 누구나 v2, v3, v10을 만들 수 있지만 해당 version은 traffic 없이 서버에 놓입니다. promotion은 production caller에 어떤 version을 전달할지 알려 주는 config를 update하는 것이며, 이 update는 일반적인 검토 절차를 거칩니다. 이 과정은 version 생성을 저렴하게 만들고 SDLC를 유지합니다. production 코드를 update하는 대신 config 값만 update하면 모든 runner가 변경을 반영하므로 여전히 이점이 있습니다.


## version 2 평가하기

같은 평가를 v2에 고정하여 수행합니다. 새 규칙은 범위가 넓고, 사용량 기반으로 청구되는 API 제품에서는 billing ticket도 API 사용량을 이야기합니다.


In [7]:
v2_scores = score(version=agent.version)

for team in sorted(v1_scores):
    c1, n1 = v1_scores[team]
    c2, n2 = v2_scores[team]
    flag = "  <-- regressed" if c2 < c1 else ""
    print(team)
    print(f"  v1: {c1}/{n1}")
    print(f"  v2: {c2}/{n2}{flag}")

api-platform
  v1: 5/5
  v2: 5/5
auth
  v1: 5/5
  v2: 5/5
billing
  v1: 4/5
  v2: 2/5  <-- regressed
dashboard
  v1: 5/5
  v2: 5/5


## rollback하기

Billing 성능이 저하되었습니다. Version 1은 여전히 서버에 있습니다. rollback은 deploy가 아닙니다. caller가 다시 `version: 1`을 전달하면 됩니다.


In [8]:
billing = [t for t in tickets if t["team"] == "billing"]
rerun = [triage(version=1, ticket=t).get("team") for t in billing]
print(f"billing tickets via version=1: {rerun.count('billing')}/{len(billing)} correct")

billing tickets via version=1: 4/5 correct


version은 서버 측에 있으므로 production을 v1에 유지하면서 PM이 v2를 계속 개선하거나 작은 비율의 traffic을 canary로 보내려 할 때도 v2는 그대로 있습니다. 수정이 준비되면 v3를 만들고 같은 절차로 promotion할 수 있습니다.


## 정리하기


In [9]:
client.beta.agents.archive(AGENT_ID)
client.beta.environments.archive(ENV_ID)
print("archived")

archived


## 되짚어 보기

여기서의 동작은 작습니다(create, update, 고정, 다시 고정). 하지만 prompt가 version이 지정된 서버 측 resource가 되면 위와 같은 `update` 실행이나 config의 version 변경을 통해 애플리케이션 코드와 독립적으로 prompt를 평가하고 promotion할 수 있습니다.

workflow에 반영할 만한 사항은 다음과 같습니다.

- production caller가 agent ID만 사용하지 않고 명시적인 version으로 고정하게 하세요. 새 version은 promotion하기 전까지 보이지 않습니다.
- 고정된 version 번호를 prompt 변경의 관문으로 취급하세요. version 생성은 탐색이며 production의 고정값을 update하는 것이 검토를 거칩니다.
- 더 중요한 agent에서는 완전한 promotion 전에 일부 traffic을 새 version으로 보내 비교하세요(이 notebook처럼). 이 versioning을 사실상 feature flag로 사용할 수 있습니다.

### 다음 단계

- `client.beta.agents.versions.list(AGENT_ID)`는 agent의 모든 version을 반환합니다.
- `managed_agents/`의 다른 notebook은 세션, custom tool, 전체 과정의 패턴을 다룹니다.

